# Transform Races Data

1. Read bronze `races` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`raceName` → `race_name`, `circuitId` → `circuit_id`)
1. Rename columns to make them more meaningful (`date` → `race_date`)
1. Remove duplicate records
1. Transform values of column `race_name` to Title Case
1. Write the transformed data to silver `races` table

Step 1 - Loading the env config from the common config

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
%run ../00-common-Config/02-helper-function

In [0]:
from pyspark.sql import functions as f

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.races"
silver_table=f"{catalog_name}.{silver_schema}.races"

In [0]:
race_df=spark.read.table(bronze_table).filter(f.col("batch_id")==v_batch_id)
display(race_df)

Step 2. Keep only the columns required for analytics (Drop url column)

In [0]:
race_selected_df=race_df.select(
    f.col("season"),
    f.col("round"),
    f.col("raceName"),
    f.col("date"),
    f.col("circuitId"),
    f.col("ingestion_timestamp"),
    f.col("source_file"),
    f.col("batch_id")
)

%md
- Standardise column names using snake_case (raceName → race_name, circuitId → circult_1d)
- Rename columns to make them more meaningful (date → race_date)

In [0]:
race_named_df=race_selected_df.withColumnsRenamed({
    "raceName":"race_name",
    "date":"race_date",
    "circuitId":"circuit_id",
})

Step 5. Remove duplicate records

In [0]:
race_distinct_df=race_named_df.dropDuplicates(["season","round"])

Step 6. Transform values of column race_name to Title Case

In [0]:
race_final_df=race_distinct_df.withColumn("race_name",f.initcap(f.col("race_name")))

In [0]:
write_to_silver(
    input_df=race_final_df,
    target_table=silver_table,
    merge_condition="t.season = s.season AND t.round = s.round",
    columns_to_update=[
        "race_name",
        "race_date",
        "circuit_id",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
# race_final_df.show()

In [0]:
# race_final_df.write\
#     .format("delta")\
#     .mode("overwrite")\
#     .saveAsTable(silver_table)

In [0]:
spark.table(silver_table).show()